In [ ]:
!pip -q install numpy
%cd /content
!rm -rf /content/DanyAI
!git clone -q https://github.com/DanyaVoredom/DanyAI.git /content/DanyAI
%cd /content/DanyAI
!pip -q install -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv || true
python -c "import torch; print('torch', torch.__version__, torch.cuda.is_available())"
print("setup done")

In [ ]:
import os, pathlib, urllib.request
try:
    os.chdir("/content/DanyAI")
except Exception:
    pass
base = "https://github.com/DanyaVoredom/DanyAI/releases/download/data-v1/"
for name in ["train.bin", "val.bin", "tokenizer.json", "meta.json", "sft.jsonl"]:
    dst = pathlib.Path("data", name)
    dst.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(base + name, dst)
    print("got", name, dst.stat().st_size)
print("data", sorted(os.listdir("data")))

In [ ]:
import json, os
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
cfg = json.load(open("configs/big.json"))
cfg.update(block_size=256, batch_size=32, steps=6000, warmup=300)
cfg["log_every"] = 10
json.dump(cfg, open("configs/colab.json", "w"), indent=2)
print(cfg, flush=True)
!python -m danyai.torch_train --config configs/colab.json --data data --out /content/danyai_runs $( [ -f /content/danyai_runs/ckpt.npz ] && echo --resume /content/danyai_runs/ckpt.npz )
!python -m danyai.sample --ckpt /content/danyai_runs/best.npz --tok data/tokenizer.json --prompt "Once upon a time" --tokens 120 --temp 0.8
print("train done")

In [ ]:
!python -m danyai.torch_sft --ckpt /content/danyai_runs/best.npz --data data/sft.jsonl --tok data/tokenizer.json --out /content/danyai_runs/sft --steps 1500 --batch 8 --lr 1e-4 --block 384
!python -m danyai.sample --ckpt /content/danyai_runs/sft/best.npz --tok data/tokenizer.json --prompt "<|user|>Придумай короткую историю про кота.<assistant|>" --tokens 120 --temp 0.7
print("sft done")

In [ ]:
import base64, pathlib
for name in ["data/tokenizer.json", "data/meta.json", "configs/colab.json"]:
    p = pathlib.Path(name)
    if not p.exists():
        print("@@MISSING", name)
        continue
    b = base64.b64encode(p.read_bytes()).decode()
    print(f"@@BEGIN {name} {len(b)}")
    for i in range(0, len(b), 50000):
        print(b[i:i + 50000])
    print(f"@@END {name}")
import base64, pathlib
p = pathlib.Path("/content/danyai_runs/best.npz")
if p.exists():
    b = base64.b64encode(p.read_bytes()).decode()
    print(f"@@BEGIN /content/danyai_runs/best.npz {len(b)}")
    for i in range(0, len(b), 50000):
        print(b[i:i + 50000])
    print("@@END /content/danyai_runs/best.npz")
else:
    print("@@MISSING /content/danyai_runs/best.npz")
print("pack a done")

In [ ]:
import base64, pathlib
p = pathlib.Path("/content/danyai_runs/sft/best.npz")
if p.exists():
    b = base64.b64encode(p.read_bytes()).decode()
    print(f"@@BEGIN /content/danyai_runs/sft/best.npz {len(b)}")
    for i in range(0, len(b), 50000):
        print(b[i:i + 50000])
    print("@@END /content/danyai_runs/sft/best.npz")
else:
    print("@@MISSING /content/danyai_runs/sft/best.npz")
print("pack done")